# 🎯 Phase 4: DPO Preference Tuning

We now take our SFT-trained model and apply DPO (Direct Preference Optimization).
DPO teaches the model to prefer correct JSON outputs over flawed ones.

Input format: (prompt, chosen_output, rejected_output) triples

**Expected training time on A100:** ~20–30 minutes
**VRAM usage:** ~24–28 GB

In [ ]:
# ── Global config ─────────────────────────────────────────────────────────────
SFT_CHECKPOINT = "./outputs/sft-qlora-checkpoint"
DPO_OUTPUT_DIR = "./outputs/dpo-checkpoint"
BASE_MODEL     = "Qwen/Qwen2.5-7B-Instruct"
WANDB_PROJECT  = "project4-lora-dpo"
MAX_SEQ_LEN    = 1024
SEED           = 42
import os, json
os.environ["HF_TOKEN"]      = "hf_XXXXXXXXXXXXXXXXXXXXXXXX"
os.environ["WANDB_API_KEY"] = "XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX"
os.makedirs(DPO_OUTPUT_DIR, exist_ok=True)

In [ ]:
# ── Load DPO dataset ──────────────────────────────────────────────────────────
from datasets import Dataset
import json

with open('./data/train_dpo.json') as f:
    train_raw = json.load(f)
with open('./data/test_dpo.json') as f:
    test_raw = json.load(f)

# DPO trainer expects: prompt, chosen, rejected columns
SYSTEM_PROMPT = """You are a precise JSON extraction assistant. 
Given unstructured text, extract the requested fields and return ONLY valid JSON.
Do not include any explanation, markdown, or extra text. Output only the JSON object."""

def format_dpo(example):
    return {
        "prompt": f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{example['prompt']}<|im_end|>\n<|im_start|>assistant\n",
        "chosen": example['chosen'],
        "rejected": example['rejected'],
    }

train_dpo = Dataset.from_list([format_dpo(ex) for ex in train_raw])
test_dpo  = Dataset.from_list([format_dpo(ex) for ex in test_raw])

print(f'DPO Train: {len(train_dpo)} | DPO Test: {len(test_dpo)}')
print('Sample keys:', train_dpo[0].keys())

In [ ]:
# ── Load SFT model + tokenizer ────────────────────────────────────────────────
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(SFT_CHECKPOINT)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # DPO requires left padding

# Load base model + SFT LoRA adapters
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model = PeftModel.from_pretrained(base, SFT_CHECKPOINT, is_trainable=True)

# Reference model (frozen copy of SFT) — required by DPO
ref_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
ref_model = PeftModel.from_pretrained(ref_base, SFT_CHECKPOINT, is_trainable=False)

print('✅ Model and reference model loaded')

In [ ]:
# ── DPO Training config ───────────────────────────────────────────────────────
from trl import DPOTrainer, DPOConfig
import wandb

wandb.init(project=WANDB_PROJECT, name="dpo-run1")

dpo_config = DPOConfig(
    output_dir=DPO_OUTPUT_DIR,
    num_train_epochs=2,               # DPO needs fewer epochs than SFT
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,    # Effective batch = 16
    learning_rate=5e-5,               # Lower LR than SFT
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    beta=0.1,                         # KL penalty weight — key DPO hyperparameter
    max_length=MAX_SEQ_LEN,
    max_prompt_length=512,
    bf16=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=100,
    save_total_limit=2,
    report_to="wandb",
    seed=SEED,
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=ref_model,
    args=dpo_config,
    train_dataset=train_dpo,
    eval_dataset=test_dpo,
    tokenizer=tokenizer,
)

print('🚀 Starting DPO training...')
dpo_trainer.train()

dpo_trainer.save_model(DPO_OUTPUT_DIR)
tokenizer.save_pretrained(DPO_OUTPUT_DIR)
print(f'✅ DPO model saved to {DPO_OUTPUT_DIR}')

In [ ]:
# ── Evaluate DPO model ────────────────────────────────────────────────────────
from tqdm import tqdm

model.eval()
tokenizer.padding_side = "right"  # Switch back for generation

def run_inference(instruction, input_text, model, tokenizer, max_new_tokens=256):
    SYSTEM_PROMPT = """You are a precise JSON extraction assistant. 
    Given unstructured text, extract the requested fields and return ONLY valid JSON.
    Do not include any explanation, markdown, or extra text. Output only the JSON object."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"{instruction}\n\nText: {input_text}"}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             temperature=0.1, do_sample=True,
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def is_valid_json(t):
    try: json.loads(t.strip()); return True
    except: return False

def exact_match(p, g):
    try: return json.loads(p.strip()) == json.loads(g.strip())
    except: return False

def field_coverage(p, g):
    try:
        po, go = json.loads(p.strip()), json.loads(g.strip())
        return sum(1 for k, v in go.items() if po.get(k) == v) / len(go)
    except: return 0.0

with open('./data/test_sft.json') as f:
    test_data = json.load(f)

results_dpo = []
for ex in tqdm(test_data[:100], desc="Evaluating DPO model"):
    pred = run_inference(ex['instruction'], ex['input_text'], model, tokenizer)
    results_dpo.append({
        'valid_json': is_valid_json(pred),
        'exact_match': exact_match(pred, ex['output']),
        'field_coverage': field_coverage(pred, ex['output']),
    })

n = len(results_dpo)
dpo_metrics = {
    'model': 'DPO Model (SFT + DPO)',
    'json_validity_rate': sum(r['valid_json'] for r in results_dpo) / n,
    'exact_match_accuracy': sum(r['exact_match'] for r in results_dpo) / n,
    'avg_field_coverage': sum(r['field_coverage'] for r in results_dpo) / n,
    'n_samples': n
}

with open('./data/dpo_metrics.json', 'w') as f:
    json.dump(dpo_metrics, f, indent=2)

print('\n✅ DPO metrics saved. Run 05_final_comparison.ipynb for the full table.')
wandb.finish()